# Vue2 + Vue3 框架 · 第 08 天

## 智慧商城项目：项目初始化、组件库、路由与登录

> 第 08～10 天使用同一套“智慧商城”项目资料。本篇整理第 08 天的基础设施和登录链路：创建项目、Vant 组件库、移动端适配、路由骨架、Axios 封装、验证码、短信倒计时、登录状态和路由拦截。

## 一、项目目标与整体架构

项目要实现一个移动端商城，功能包括首页、搜索、商品详情、购物车、订单结算、个人中心和登录。第 08 天先把“能运行、能请求、能登录、能保护页面”的基础架子搭好。

```text
src/
├─ api/          接口模块
├─ assets/       图片和全局样式
├─ components/   通用组件
├─ router/       路由配置
├─ store/        Vuex 状态
├─ utils/        request、工具方法
├─ views/        页面级组件
├─ App.vue
└─ main.js
```

设计原则：页面放 `views`，复用 UI 放 `components`，请求地址集中放 `api`，Axios 实例集中放 `utils/request.js`，登录 token 和购物车等跨页面状态放 Vuex。

## 二、创建项目与初始化目录

```bash
vue create hm-shopping
cd hm-shopping
npm run serve
```

清理脚手架默认示例后保留 `App.vue`、`main.js`、`public/index.html` 和必要的 `assets`。创建目录时不要一开始把所有业务都塞到 `App.vue`，后续每个页面都应该成为独立的路由组件。

### 安装项目依赖

```bash
npm install axios vue-router@3 vuex@3 vant@2
npm install postcss-px-to-viewport --save-dev
```

课程项目使用 Vue 2 + Vant 2；如果新建 Vue 3 项目，应使用 Vue Router 4、Pinia 和 Vant 4，概念相同但安装包版本不同。

## 三、Vant 组件库与按需导入

### 全部导入

```js
import Vue from 'vue'
import Vant from 'vant'
import 'vant/lib/index.css'
Vue.use(Vant)
```

简单但会把全部组件打进首屏包。

### 按需导入

```js
import Vue from 'vue'
import { Button, Toast, NavBar, Tabbar, TabbarItem } from 'vant'

Vue.use(Button)
Vue.use(NavBar)
Vue.use(Tabbar)
Vue.use(TabbarItem)
Vue.use(Toast)
```

按需导入减少首屏体积，业务项目推荐结合构建工具使用。使用前检查组件是否完成 `Vue.use`，否则模板标签可能无法渲染。

## 四、移动端 `vw` 适配

项目设计稿通常以 375px 宽度为基准。通过 `postcss-px-to-viewport` 将 CSS 中的 px 转换为 vw：

```js
// vue.config.js
module.exports = {
  css: {
    loaderOptions: {
      postcss: {
        postcssOptions: {
          plugins: [
            require('postcss-px-to-viewport')({
              viewportWidth: 375,
              unitPrecision: 5,
              viewportUnit: 'vw',
              selectorBlackList: ['ignore'],
              minPixelValue: 1
            })
          ]
        }
      }
    }
  }
}
```

适配插件只处理样式单位，图片、接口和组件逻辑仍然需要单独处理。第三方组件如果不希望转换，可按插件规则加入忽略类名。

## 五、路由骨架与 Tabbar

### 5.1 一级路由

```js
// router/index.js
import Vue from 'vue'
import VueRouter from 'vue-router'
import Layout from '@/views/Layout.vue'
import Login from '@/views/login/index.vue'

Vue.use(VueRouter)
export default new VueRouter({
  routes: [
    { path: '/', component: Layout },
    { path: '/login', component: Login }
  ]
})
```

### 5.2 Tabbar 页面

`Layout.vue` 负责公共底部导航和页面出口：

```vue
<template>
  <div class="layout">
    <router-view />
    <van-tabbar route>
      <van-tabbar-item to="/home" icon="home-o">首页</van-tabbar-item>
      <van-tabbar-item to="/category" icon="apps-o">分类</van-tabbar-item>
      <van-tabbar-item to="/cart" icon="shopping-cart-o">购物车</van-tabbar-item>
      <van-tabbar-item to="/user" icon="user-o">我的</van-tabbar-item>
    </van-tabbar>
  </div>
</template>
```

页面之间只替换 `router-view`，底部 Tabbar 保持不变；需要子页面时再在对应页面下配置二级路由。

## 六、登录页与 Axios 请求封装

### 6.1 `request.js`

```js
import axios from 'axios'
import store from '@/store'
import router from '@/router'
import { Toast } from 'vant'

const request = axios.create({
  baseURL: 'http://cba.itlike.com',
  timeout: 10000
})

request.interceptors.request.use(config => {
  const token = store.getters.token
  if (token) config.headers.Authorization = `Bearer ${token}`
  return config
})

request.interceptors.response.use(
  response => response.data,
  error => {
    Toast.fail(error.response?.data?.message || '请求失败')
    if (error.response?.status === 401) router.replace('/login')
    return Promise.reject(error)
  }
)

export default request
```

请求模块统一处理 baseURL、超时、token 和错误提示，页面只关注接口参数和业务结果。

### 6.2 验证码和登录 API

```js
// api/user.js
import request from '@/utils/request'
export const getPicCode = () => request.get('/captcha/image')
export const getMsgCode = (mobile, captcha) => request.post('/captcha/sendSms', { mobile, captcha })
export const login = data => request.post('/user/login', data)
```

## 七、短信倒计时与表单校验

```js
export default {
  data () {
    return { mobile: '', picCode: '', msgCode: '', countdown: 0, timer: null }
  },
  methods: {
    async sendMsg () {
      if (!/^1[3-9]\d{9}$/.test(this.mobile)) return this.$toast('请输入正确手机号')
      await getMsgCode(this.mobile, this.picCode)
      this.countdown = 60
      this.timer = setInterval(() => {
        this.countdown--
        if (this.countdown <= 0) {
          clearInterval(this.timer)
          this.timer = null
        }
      }, 1000)
    }
  },
  beforeDestroy () {
    clearInterval(this.timer)
  }
}
```

发送按钮显示 `countdown > 0 ? countdown + '秒后重试' : '获取验证码'`，请求期间禁用按钮，避免重复请求。所有输入在提交前再次校验，不能只依靠按钮的 disabled 状态。

## 八、登录状态、持久化与访问拦截

登录成功后把服务端返回的 token 存入 Vuex：

```js
// store/modules/user.js
const user = {
  namespaced: true,
  state: () => ({ token: localStorage.getItem('token') || '' }),
  mutations: {
    setToken (state, token) {
      state.token = token
      token ? localStorage.setItem('token', token) : localStorage.removeItem('token')
    }
  },
  getters: { token: state => state.token }
}
export default user
```

路由前置守卫保护需要登录的页面：

```js
router.beforeEach((to, from, next) => {
  const token = store.getters['user/token']
  if (to.meta.requireAuth && !token) {
    return next({ path: '/login', query: { backUrl: to.fullPath } })
  }
  next()
})
```

登录成功后使用 `backUrl` 返回用户原来想访问的页面；退出时清空 Vuex、localStorage 和购物车。

## 九、请求 Loading 与今天的检查清单

可以用请求拦截器统一维护 loading 次数，或在页面中使用 `v-loading`；注意并发请求时不能简单用一个布尔值覆盖掉仍在进行的请求。

完成第 08 天后应能回答：

- 项目入口、页面、组件、接口和状态分别放在哪里？
- Vant 全量导入和按需导入有什么差异？
- token 如何保存并自动放到请求头？
- 未登录访问受保护页面如何跳到登录并返回原页面？
- 验证码、短信倒计时和登录错误提示如何复用？